In [ ]:
import numpy as np
import torch

DATA_DIR = "data/gutenberg_dataset"

train_path = f"{DATA_DIR}/train.bin"
val_path   = f"{DATA_DIR}/val.bin"
test_path  = f"{DATA_DIR}/test.bin"


In [ ]:
train_data = np.memmap(
    train_path,
    dtype=np.uint16,
    mode="r"
)

val_data = np.memmap(
    val_path,
    dtype=np.uint16,
    mode="r"
)

test_data = np.memmap(
    test_path,
    dtype=np.uint16,
    mode="r"
)

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))
print("Test tokens:", len(test_data))

In [ ]:
from torch.utils.data import IterableDataset, DataLoader

class GutenbergDataset(IterableDataset):

    def __init__(
        self,
        data,
        seq_len,
        samples_per_epoch,
    ):
        super().__init__()

        self.data = data
        self.seq_len = seq_len
        self.samples_per_epoch = (
            samples_per_epoch
        )

    def __iter__(self):

        worker_info = (
            torch.utils.data.get_worker_info()
        )

        if worker_info is None:
            worker_id = 0
            num_workers = 1
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        rng = np.random.default_rng(
            42 + worker_id
        )

        max_start = (
            len(self.data)
            - self.seq_len
            - 1
        )

        n = (
            self.samples_per_epoch
            // num_workers
        )

        for _ in range(n):

            start = rng.integers(
                0,
                max_start
            )

            # One copy
            chunk = np.asarray(
                self.data[
                    start:
                    start + self.seq_len + 1
                ],
                dtype=np.int64
            )

            chunk = torch.from_numpy(
                chunk
            )

            yield (
                chunk[:-1],
                chunk[1:]
            )

In [ ]:
batch_size = 64
seq_len = 512

tokens_per_batch = (
    batch_size * seq_len
)

# Let's process approximately 1M tokens per epoch
tokens_per_epoch = 1_000_000

steps_per_epoch = (
    tokens_per_epoch
    // tokens_per_batch
)

samples_per_epoch = (
    steps_per_epoch
    * batch_size
)

print(
    "Tokens/batch:",
    tokens_per_batch
)

print(
    "Steps/epoch:",
    steps_per_epoch
)

print(
    "Samples/epoch:",
    samples_per_epoch
)

In [ ]:
import torch.multiprocessing as mp

train_dataset = GutenbergDataset(
    train_data,
    seq_len=512,
    samples_per_epoch=samples_per_epoch
)

test_dataset = GutenbergDataset(
    test_data,
    seq_len=512,
    samples_per_epoch=samples_per_epoch
)

val_dataset = GutenbergDataset(
    val_data,
    seq_len=512,
    samples_per_epoch=samples_per_epoch
)


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
)

xb, yb = next(iter(train_loader))

#######
print("X shape:", xb.shape)
print("Y shape:", yb.shape)

print("X dtype:", xb.dtype)
print("Y dtype:", yb.dtype)

print("First 20 X tokens:")
print(xb[0, :20])

print()

print("First 20 Y tokens:")
print(yb[0, :20])

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    f"{DATA_DIR}/tokenizer.json"
)

vocab_size = tokenizer.get_vocab_size()

######
tokens = xb[0].tolist()

text = tokenizer.decode(
    tokens
)

print(text)

In [ ]:
from torch import nn
from torch.nn import functional as F
import math

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self,head_dim, max_seq_len = 4096, base = 10000) -> None:
        super().__init__()
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.base = base
        # Frequencies for each pair of dimensions.
        #
        # Shape:
        # (head_dim / 2)
        #
        inv_freq = 1.0 / (
            base ** (torch.arange(0,head_dim,2).float()/ head_dim)
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self,seq_len):
        device = self.inv_freq.device
        positions = torch.arange(
            seq_len,
            device = device,
            dtype=self.inv_freq.dtype
        )

        freqs = torch.outer(positions,self.inv_freq)

        # Duplicate frequencies so that:
        # [freq0, freq1, freq2]
        # becomes
        # [freq0, freq0, freq1, freq1, freq2, freq2]
        # matching pairs of dimensions.

        emb = torch.cat([freqs, freqs],dim=-1)

        cos = emb.cos()
        sin = emb.sin()

        # Shape:
        # (1, 1, seq_len, head_dim)

        self.register_buffer(
            "cos_cached",
            cos[None, None, :, :],
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            sin[None, None, :, :],
            persistent=False
        )

    def rotate_half(self,x):
        # Split:
        # [x0 x1 x2 x3]
        # into:
        # [x0 x1] [x2 x3]
        # and rotate each pair.

        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]

        return torch.cat(
            (-x2, x1),
            dim=-1
        )
    
    def forward(self, x, positions):
        # x:
        # (B, heads, T, head_dim)
        # positions:
        # (T,)

        max_position = positions.max().item()

        if max_position >= self.cos_cached.shape[2]:
            self._build_cache(
                max_position + 1
            )

        cos = self.cos_cached[:,:,positions,:]
        sin = self.sin_cached[:,:,positions,:]

        return (
            x * cos
            + self.rotate_half(x) * sin
        )

In [ ]:
from torch.nn.attention import sdpa_kernel, SDPBackend

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embed,n_head,n_kv_head,Dropout=0.0) -> None:
        super().__init__()

        assert n_head % n_kv_head == 0
        assert n_embed % n_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head

        self.head_dim = n_embed // n_head

        self.q_proj = nn.Linear(n_embed,n_head * self.head_dim,bias=False)
        self.k_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)
        self.v_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)

        self.proj = nn.Linear(n_embed, n_embed,bias=False)
        self.dropout_p = Dropout

        self.rope = RotaryEmbedding(head_dim=self.head_dim,max_seq_len=4096)


    def forward(self,x,past_k = None,past_v = None, use_cache = False):
        B, T, C = x.shape

        #Q
        q =self.q_proj(x)
        q = q.view(B,T,self.n_head,self.head_dim)
        q = q.transpose(1,2)
        # (B, n_head, T, head_dim)

        # K
        k = self.k_proj(x)
        k = k.view(B,T,self.n_kv_head,self.head_dim)
        k = k.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #V
        v = self.v_proj(x)
        v = v.view(B,T,self.n_kv_head,self.head_dim)
        v = v.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #RoPE
        if past_k is None:
            # Training / prefill.
            # Current tokens occupy:
            # 0, 1, 2, ..., T-1
            past_len = 0
        else:
            # During generation:
            # cached tokens already occupy:
            # 0 ... past_len-1
            past_len = past_k.size(2)

        positions = torch.arange(past_len,past_len + T,device=x.device)

        q = self.rope(q,positions)
        k = self.rope(k,positions)
        
        #KV Cache
        if past_k is not None:
            k = torch.cat([past_k, k],dim=2)

        if past_v is not None:
            v = torch.cat([past_v, v],dim=2)

        #Att
        if past_k is None:
            # Training / prefill
            # Q length == K/V length
            # Therefore normal causal attention.
            with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
                out = F.scaled_dot_product_attention(q, k, v, 
                is_causal=True,
                dropout_p=(self.dropout_p if self.training else 0.0)
                , enable_gqa=True
                )

        else:
            # Decode with KV cache
            # q = only the NEW token
            # k/v = entire cache
            # The new token should attend to ALL
            # cached tokens, so DO NOT use is_causal=True.
            with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
                out = F.scaled_dot_product_attention(q,k,v,
                    is_causal=False,
                    dropout_p=0.0,
                    enable_gqa=True
                )

        out = out.transpose(1, 2)

        out = out.contiguous().view(
            B,
            T,
            C
        )

        out = self.proj(out)

        if use_cache:
            return out, k, v

        return out


In [ ]:
class FeedForward(nn.Module):
    def __init__(self,n_embed,dropout_p) -> None:
        super().__init__()
        # Standard Transformer FFN:
        #
        # 4 * n_embed
        #
        # SwiGLU has two input projections, so we reduce
        # the hidden size to keep parameter count roughly
        # comparable to a normal 4x FFN.

        hidden_dim = int(
            (8 / 3) * n_embed
        )
        hidden_dim = (
            (hidden_dim + 7) // 8
        ) *  8
        self.w1 = nn.Linear(n_embed,hidden_dim,bias=False        )
        # x -> W2
        self.w2 = nn.Linear(n_embed,hidden_dim,bias=False)
        # output projection
        self.w3 = nn.Linear(hidden_dim,n_embed,bias=False)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        # First branch:
        # Swish(xW1)
        gate = F.silu(self.w1(x))
        # Second branch:
        value = self.w2(x)
        # Elementwise gating:
        # Swish(xW1) * (xW2)
        x = gate * value
        # Output projection
        x = self.w3(x)
        return self.dropout(x)

In [ ]:
class Block(nn.Module):
    def __init__(self,n_embed,n_head,n_kv_head,dropout_p) -> None:
        super().__init__()

        self.sa = MultiHeadAttention(
            n_embed=n_embed,
            n_head=n_head,
            n_kv_head=n_kv_head,
            Dropout=dropout_p
        )

        self.ffwd = FeedForward(n_embed, dropout_p)
        self.norm1 = nn.RMSNorm(n_embed)
        self.norm2 = nn.RMSNorm(n_embed)

    def forward(self,x,past_k = None, past_v = None, use_cache = False):
        x_norm = self.norm1(x)

        if use_cache:
            attn_out,k,v = self.sa(
                x_norm,
                past_k = past_k,
                past_v = past_v,
                use_cache = True
            )
        else:
            attn_out = self.sa(x_norm)

        x = x + attn_out
        x = x + self.ffwd(self.norm2(x))

        if use_cache:
            return x, k, v

        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self,n_embed,n_head,n_layer,n_kv_head,dropout_p) -> None:
        super().__init__()
        self.blocks = nn.ModuleList([
            Block(n_embed,n_head,n_kv_head,dropout_p) for _ in range(n_layer)
        ])
        self.token_embedding = nn.Embedding(vocab_size,n_embed)
        self.ln_f = nn.RMSNorm(n_embed)
        self.lm_head = nn.Linear(n_embed,vocab_size)
        self.lm_head.weight = self.token_embedding.weight
        
    def forward(self,x, targets = None, past_kvs = None, use_cache = False):
        B,T = x.shape
        x = self.token_embedding(x)

        # Transformer Blocks
        new_kvs = []

        for i, block in enumerate(self.blocks):
            if use_cache:
                past_k = None
                past_v = None

                if past_kvs is not None:
                    past_k, past_v = past_kvs[i]

                x,k,v = block(x, past_k = past_k, past_v = past_v, use_cache = True)
                new_kvs.append((k,v))
            else:
                x = block(x)
        
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B,T,C = logits.shape

            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B*T)

            loss = F.cross_entropy(logits_flat,targets_flat)
        if use_cache:
            return logits, loss, new_kvs

        return logits, loss

In [ ]:
batch_size = 64
block_size = seq_len

n_embed = 256
n_head = 8
n_kv_head = 4
n_layer = 10

dropout = 0.1
learning_rate = 3e-4

# Trial Model
warmup_steps = 500
max_steps = 10000

min_lr = 3e-5
eval_interval = 500

assert n_embed % n_head == 0
assert n_head % n_kv_head == 0

head_dim = n_embed // n_head
assert head_dim % 2 == 0

In [ ]:
print(torch.backends.cuda.is_flash_attention_available())
print(torch.backends.cuda.mem_efficient_sdp_enabled())
print(torch.backends.cuda.math_sdp_enabled())

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Transformer(
    n_embed=n_embed,
    n_head=n_head,
    n_layer=n_layer,
    n_kv_head=n_kv_head,
    dropout_p=dropout
).to(device)


In [ ]:
def get_lr(step):

    # Warmup
    if step < warmup_steps:

        return learning_rate * (
            step + 1
        ) / warmup_steps

    # Cosine decay
    decay_ratio = (
        step - warmup_steps
    ) / (
        max_steps - warmup_steps
    )

    decay_ratio = min(
        decay_ratio,
        1.0
    )

    coeff = 0.5 * (
        1.0 + math.cos(
            math.pi * decay_ratio
        )
    )

    return min_lr + coeff * (
        learning_rate - min_lr
    )


In [ ]:
num_params = sum(
    p.numel()
    for p in model.parameters()
    )

print(f"Parameters: {num_params:,}")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.1,
    betas=(0.9, 0.95)
)

scaler = torch.amp.GradScaler("cuda")

In [ ]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    total_loss = 0.0
    total_batches = 0

    for xb, yb in val_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(
                xb,
                yb
            )

        total_loss += loss.item()
        total_batches += 1

    model.train()

    return total_loss / total_batches

In [ ]:

global_step = 0

train_loss_accum = 0.0
train_loss_count = 0

model.train()

while global_step < max_steps:

    for xb, yb in train_loader:

        if global_step >= max_steps:
            break

        xb = xb.to(device)
        yb = yb.to(device)

        # ---------------------------
        # Learning rate
        # ---------------------------

        lr = get_lr(global_step)

        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # ---------------------------
        # Forward
        # ---------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(
                xb,
                yb
            )

        # ---------------------------
        # Backward
        # ---------------------------

        scaler.scale(loss).backward()

        # Unscale gradients
        scaler.unscale_(optimizer)

        # Gradient clipping
        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )
        )

        # ---------------------------
        # Optimizer step
        # ---------------------------

        scaler.step(optimizer)
        scaler.update()

        # ---------------------------
        # Track training loss
        # ---------------------------

        train_loss_accum += loss.item()
        train_loss_count += 1

        global_step += 1

        # ---------------------------
        # Validation
        # ---------------------------

        if global_step % eval_interval == 0:

            avg_train_loss = (
                train_loss_accum
                / train_loss_count
            )

            val_loss = estimate_loss()

            print(
                f"Step {global_step:5d} "
                f"| Train Loss: {avg_train_loss:.4f} "
                f"| Val Loss: {val_loss:.4f} "
                f"| LR: {lr:.2e} "
                f"| Grad Norm: {grad_norm:.2f}"
            )

            train_loss_accum = 0.0
            train_loss_count = 0

In [ ]:
model.eval()

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

past_kvs = None

with torch.no_grad():

    for step in range(256):

        if past_kvs is None:

            # First pass:
            # process the entire current context

            context_cond = context[:, -block_size:]

            logits, _, past_kvs = model(
                context_cond,
                use_cache=True
            )

        else:

            # Subsequent passes:
            # ONLY process the new token

            new_token = context[:, -1:]

            logits, _, past_kvs = model(
                new_token,
                past_kvs=past_kvs,
                use_cache=True
            )

        # Last token's logits

        logits = logits[:, -1, :]

        # Temperature can be added here
        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        context = torch.cat(
            (context, next_token),
            dim=1
        )

print(
    tokenizer.decode(
        context[0].tolist()
    )
)